# Smoke test — does everything work?

Run the cells top to bottom (Kernel → Restart & Run All). Each step prints **OK**, or **SKIPPED**
with the reason if an external tool is not installed. It takes well under a minute; nothing heavy
(no full CFD solve) is run.

In [ ]:
from pathlib import Path
from vegeta import dedalus, talos, aeromant, mellonia

RUNS = Path("_runs/smoke"); RUNS.mkdir(parents=True, exist_ok=True)
print("vegeta tools imported:", dedalus.__version__, talos.__version__, aeromant.__version__, mellonia.__version__)

## 1. Dedalus — make a small beam and export STEP + STL

In [ ]:
from vegeta.dedalus.examples import CantileverBeam

beam = CantileverBeam().generate(length=100, width=10, height=10)
cad = beam.export(RUNS / "cad")
print("OK" if cad.ok else "FAILED", "- volume", beam.volume, "mm^3 (expected 10000)")
beam

## 2. Talos — mesh and solve the beam (needs Gmsh + CalculiX)

In [ ]:
model = talos.StructuralModel(
    geometry=cad.artifacts["step"], units="mm-N-MPa",
    material=talos.Material("steel", youngs_modulus=210000, poissons_ratio=0.3, yield_strength=235),
    regions=[talos.SurfacesOnPlane("fixed", "x", 0), talos.SurfacesOnPlane("tip", "x", 100)],
    supports=[talos.FixedSupport("fixed")],
    loads=[talos.Force("tip", fz=-100)],
    mesh_settings=talos.MeshSettings(element_size=5),
)
mesh = model.mesh(RUNS / "fea")
fea = model.solve(RUNS / "fea") if mesh.ok else mesh
if fea.ok:
    theory = 100 * 100**3 / (3 * 210000 * 10 * 10**3 / 12)
    print(f"OK - tip deflection {-fea.metrics['displacement_min'][2]:.4f} mm (beam theory {theory:.4f} mm)")
else:
    print("SKIPPED/FAILED:", fea.messages)

## 3. Mellonia — slice the STL (needs PrusaSlicer)

In [ ]:
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

prn = mellonia.slice_stl(cad.artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(), RUNS / "print")
if prn.ok:
    m = prn.metrics
    print(f"OK - {m['layer_count']} layers, {m['estimated_time']}, {m['filament_used_g']} g")
else:
    print("SKIPPED/FAILED:", prn.messages)

## 4. Aeromant — prepare a case and build the background mesh only (needs OpenFOAM)

In [ ]:
import math

try:
    env = aeromant.OpenFOAMEnvironment.detect()
except RuntimeError as exc:
    env = None
    print("SKIPPED:", exc)

if env is not None:
    case = aeromant.CFDCase(
        "laminar_external_simplefoam", cad.artifacts["stl"],
        dict(velocity=1.0, kinematic_viscosity=1e-3, density=1.0, reference_area=1e-4,
             reference_length=0.1, center_of_rotation=(0, 0, 0)),
        workdir=RUNS / "cfd", geometry_units="mm", environment=env,
    )
    prep = case.prepare(overwrite=True)
    run = case.run(steps=["blockMesh", "checkMesh"]) if prep.ok else prep
    print("OK - background mesh cells:", run.metrics.get("mesh_cells") if run.ok else run.messages)

## 5. Vegeta Core — workspace, revision, status (needs vegeta-core)

In [ ]:
import shutil
try:
    from vegeta import core
    shutil.rmtree(RUNS / "ws", ignore_errors=True)
    ws = core.Workspace.create(RUNS / "ws")
    r1 = ws.add_design("beam", "vegeta.dedalus.examples:CantileverBeam").new_revision(length=100.0)
    gen = r1.generate()
    r2 = r1.branch(height=12.0)
    print("OK" if gen.ok else "FAILED", "-", "r2 CAD is", ws.status().rows[1]["CAD"])
    print(ws.status())
except ImportError:
    gen = None
    print("SKIPPED: vegeta-core is not installed")

## 6. Summary

In [ ]:
for name, res in [("dedalus", cad), ("talos", fea), ("mellonia", prn)] + ([("aeromant", run)] if env else []) + ([("core", gen)] if gen else []):
    print(f"{name:9s} {res.status.upper()}")